# Ollama (Native / OpenAI / Anthropic Compatible)

Ollama server behind Caddy auth (Basic Auth + Bearer + Anthropic `x-api-key`), with automatic model download & GPU VRAM warmup (`--models`, `--alias`, `--preload`), selectable or full-open compatible APIs (`--compat all|ollama|openai|anthropic`), and full `OLLAMA_*` server optimization tuning.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "ollama")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "11434"))
AUTH         = os.environ.get("CT_AUTH", "caddy")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "18080"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- install Ollama and load runtime configuration (idempotent)
from pathlib import Path
(Path(WORK) / "ollama_runtime.py").write_text('"""Shared configuration, Caddy routing, and model management helpers for the Ollama Colab service.\n\nImported both by ``src/colabllm/services/ollama.py`` on the local CLI side (standard library only)\nand embedded into ``src/colabllm/services/ollama.ipynb`` on the Colab VM.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport re\nimport time\nimport urllib.error\nimport urllib.request\nfrom typing import Any, Callable\n\nDEFAULT_MODEL = "qwen2.5:0.5b"\nDEFAULT_KEEP_ALIVE = "30m"\n\nCOMPAT_APIS = ("ollama", "openai", "anthropic")\nCOMPAT_ALIASES = {\n    "ollama": "ollama",\n    "native": "ollama",\n    "api": "ollama",\n    "openai": "openai",\n    "oai": "openai",\n    "v1": "openai",\n    "anthropic": "anthropic",\n    "claude": "anthropic",\n}\n\nKV_CACHE_TYPES = ("f16", "q8_0", "q4_0")\nMODELS_STORAGE_MODES = ("ssd", "drive")\n\n# Presets grounded in https://docs.ollama.com/faq and https://docs.ollama.com/context-length\nOPTIMIZATION_PROFILES: dict[str, dict[str, str]] = {\n    "balanced": {\n        "OLLAMA_KEEP_ALIVE": DEFAULT_KEEP_ALIVE,\n    },\n    "low-vram": {\n        "OLLAMA_FLASH_ATTENTION": "1",\n        "OLLAMA_KV_CACHE_TYPE": "q8_0",\n        "OLLAMA_NUM_PARALLEL": "1",\n        "OLLAMA_MAX_LOADED_MODELS": "1",\n        "OLLAMA_KEEP_ALIVE": DEFAULT_KEEP_ALIVE,\n    },\n    "coding": {\n        "OLLAMA_CONTEXT_LENGTH": "65536",\n        "OLLAMA_FLASH_ATTENTION": "1",\n        "OLLAMA_KV_CACHE_TYPE": "q8_0",\n        "OLLAMA_NUM_PARALLEL": "1",\n        "OLLAMA_MAX_LOADED_MODELS": "1",\n        "OLLAMA_KEEP_ALIVE": "-1",\n    },\n    "agent": {\n        "OLLAMA_CONTEXT_LENGTH": "65536",\n        "OLLAMA_FLASH_ATTENTION": "1",\n        "OLLAMA_KV_CACHE_TYPE": "q8_0",\n        "OLLAMA_NUM_PARALLEL": "1",\n        "OLLAMA_MAX_LOADED_MODELS": "1",\n        "OLLAMA_KEEP_ALIVE": "-1",\n    },\n    "throughput": {\n        "OLLAMA_FLASH_ATTENTION": "1",\n        "OLLAMA_NUM_PARALLEL": "4",\n        "OLLAMA_MAX_QUEUE": "512",\n        "OLLAMA_KEEP_ALIVE": "-1",\n    },\n}\n\nALLOWED_EXTRA_ENV_EXACT = {\n    "CUDA_VISIBLE_DEVICES",\n    "ROCR_VISIBLE_DEVICES",\n    "HIP_VISIBLE_DEVICES",\n    "GGML_VK_VISIBLE_DEVICES",\n    "GPU_DEVICE_ORDINAL",\n    "HSA_OVERRIDE_GFX_VERSION",\n    "HTTPS_PROXY",\n    "https_proxy",\n    "NO_PROXY",\n    "no_proxy",\n    "JETSON_JETPACK",\n}\nFORBIDDEN_ENV_KEYS = {\n    "OLLAMA_HOST",   # Must always bind to 127.0.0.1:<SERVICE_PORT> behind Caddy\n    "HTTP_PROXY",    # Ollama docs warn HTTP_PROXY breaks local client connections; use HTTPS_PROXY\n    "http_proxy",\n}\n\nMODEL_SPEC_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._:/@-]{0,254}$")\nDURATION_RE = re.compile(r"^-?(?:\\d+(?:\\.\\d+)?(?:ns|us|µs|ms|s|m|h))+$|^-?\\d+$")\nBYTE_SIZE_RE = re.compile(r"^\\s*(\\d+(?:\\.\\d+)?)\\s*([kmgt]i?b?)?\\s*$", re.IGNORECASE)\nVERSION_RE = re.compile(r"^[0-9]+\\.[0-9]+\\.[0-9]+(?:-[A-Za-z0-9._-]+)?$")\n\n# Caddy path patterns per API family (zero collision across /api/*, OpenAI /v1/*, Anthropic /v1/messages*)\nAPI_FAMILY_PATHS: dict[str, tuple[str, ...]] = {\n    "ollama": ("/api/*",),\n    "openai": (\n        "/v1/chat/completions",\n        "/v1/completions",\n        "/v1/responses",\n        "/v1/embeddings",\n        "/v1/models",\n        "/v1/models/*",\n        "/v1/audio/*",\n    ),\n    "anthropic": (\n        "/v1/messages",\n        "/v1/messages/*",\n        "/v1/models",\n        "/v1/models/*",\n    ),\n}\nMODEL_MUTATION_PATHS = (\n    "/api/pull",\n    "/api/push",\n    "/api/create",\n    "/api/copy",\n    "/api/delete",\n    "/api/blobs/*",\n)\n\n\ndef validate_model_spec(spec: str) -> str:\n    """Validate an Ollama library or Hugging Face model tag, rejecting traversal and CLI flags."""\n    cleaned = (spec or "").strip()\n    if not cleaned or ".." in cleaned or not MODEL_SPEC_RE.fullmatch(cleaned):\n        raise ValueError(\n            f"Invalid model name {spec!r}: use a valid Ollama model tag (e.g. qwen3:8b) or a Hugging Face GGUF reference (e.g. hf.co/org/repo:Q4_K_M)"\n        )\n    return cleaned\n\n\ndef parse_model_specs(\n    raw_values: list[str] | str | None,\n    *,\n    no_models: bool = False,\n    default: str | None = DEFAULT_MODEL,\n) -> list[str]:\n    """Parse repeatable/comma-separated model specs into an ordered deduplicated list."""\n    if isinstance(raw_values, str):\n        items = [raw_values]\n    elif raw_values is None:\n        items = []\n    else:\n        items = list(raw_values)\n\n    if no_models:\n        if any(part.strip() for item in items for part in str(item).split(",")):\n            raise ValueError("--no-models cannot be combined with a non-empty --models")\n        return []\n\n    if raw_values is None:\n        return [validate_model_spec(default)] if default else []\n\n    tokens: list[str] = []\n    explicit_none = False\n    for item in items:\n        for part in str(item).split(","):\n            tok = part.strip()\n            if not tok:\n                continue\n            if tok.lower() in ("none", "off"):\n                explicit_none = True\n                continue\n            tokens.append(validate_model_spec(tok))\n\n    if explicit_none and tokens:\n        raise ValueError("The model list cannot contain both \'none\' and specific model names")\n    if explicit_none or not tokens:\n        return []\n    return list(dict.fromkeys(tokens))\n\n\ndef parse_model_aliases(raw_values: list[str] | str | None) -> list[tuple[str, str]]:\n    """Parse repeatable/comma-separated ``ALIAS=SOURCE_MODEL`` mappings."""\n    if not raw_values:\n        return []\n    items = [raw_values] if isinstance(raw_values, str) else list(raw_values)\n    mapping: dict[str, str] = {}\n    for item in items:\n        for part in str(item).split(","):\n            entry = part.strip()\n            if not entry:\n                continue\n            alias_raw, sep, source_raw = entry.partition("=")\n            if not sep:\n                raise ValueError(f"--alias must have the form ALIAS=MODEL (e.g. gpt-4o=qwen3:8b), got: {entry!r}")\n            alias = validate_model_spec(alias_raw)\n            source = validate_model_spec(source_raw)\n            if alias == source:\n                raise ValueError(f"A model alias cannot be the same as its source model: {alias!r}")\n            if alias in mapping and mapping[alias] != source:\n                raise ValueError(f"Model alias {alias!r} is mapped to different models: {mapping[alias]!r} and {source!r}")\n            mapping[alias] = source\n    return list(mapping.items())\n\n\ndef models_to_pull(models: list[str], aliases: list[tuple[str, str]]) -> list[str]:\n    """Return ordered deduplicated list of models to download, including alias source models."""\n    combined = list(models)\n    alias_names = {alias for alias, _ in aliases}\n    for _, source in aliases:\n        if source not in combined and source not in alias_names:\n            combined.append(source)\n    return list(dict.fromkeys(combined))\n\n\ndef parse_compat_apis(raw_values: list[str] | str | None) -> tuple[str, ...]:\n    """Normalize compatible API selection into canonical order ``(\'ollama\', \'openai\', \'anthropic\')``."""\n    if not raw_values:\n        return COMPAT_APIS\n    items = [raw_values] if isinstance(raw_values, str) else list(raw_values)\n    selected: set[str] = set()\n    for item in items:\n        for part in str(item).split(","):\n            tok = part.strip().lower()\n            if not tok:\n                continue\n            if tok in ("all", "*"):\n                return COMPAT_APIS\n            canonical = COMPAT_ALIASES.get(tok)\n            if not canonical:\n                raise ValueError(\n                    f"Unsupported compatibility API {part.strip()!r}; choices: all, ollama, openai, anthropic (comma-separated)"\n                )\n            selected.add(canonical)\n    if not selected:\n        raise ValueError("--compat needs at least one API (all, ollama, openai, anthropic)")\n    return tuple(api for api in COMPAT_APIS if api in selected)\n\n\ndef validate_duration(value: str, flag_name: str = "duration") -> str:\n    """Validate Go duration strings or integer seconds accepted by Ollama (e.g. 30m, 24h, -1, 0)."""\n    cleaned = str(value).strip()\n    if not cleaned or not DURATION_RE.fullmatch(cleaned):\n        raise ValueError(\n            f"Invalid {flag_name} ({value!r}): use a Go duration (e.g. 30m, 24h, 5m) or integer seconds (-1 keeps the model loaded forever, 0 unloads immediately / no limit)"\n        )\n    return cleaned\n\n\ndef normalize_api_keep_alive(value: str | int | None) -> str | int | None:\n    """Convert unitless integer strings (e.g. \'-1\', \'0\', \'3600\') to int for Ollama JSON Duration unmarshaling."""\n    if value is None:\n        return None\n    if isinstance(value, int) and not isinstance(value, bool):\n        return value\n    cleaned = validate_duration(str(value), "keep_alive")\n    if re.fullmatch(r"-?\\d+", cleaned):\n        return int(cleaned)\n    return cleaned\n\n\ndef parse_byte_size(value: str | int, flag_name: str = "--gpu-overhead") -> int:\n    """Parse integer bytes or human size strings (e.g. 512M, 1.5G, 1073741824) into non-negative bytes."""\n    if isinstance(value, int) and not isinstance(value, bool):\n        if value < 0:\n            raise ValueError(f"{flag_name} cannot be negative")\n        return value\n    raw = str(value).strip()\n    match = BYTE_SIZE_RE.fullmatch(raw)\n    if not match:\n        raise ValueError(f"Invalid {flag_name} ({value!r}): use a non-negative byte count or a size with a unit (e.g. 512M, 1G)")\n    number = float(match.group(1))\n    unit = (match.group(2) or "").lower()\n    multipliers = {\n        "": 1,\n        "b": 1,\n        "k": 1024, "kb": 1024, "kib": 1024,\n        "m": 1024**2, "mb": 1024**2, "mib": 1024**2,\n        "g": 1024**3, "gb": 1024**3, "gib": 1024**3,\n        "t": 1024**4, "tb": 1024**4, "tib": 1024**4,\n    }\n    if unit not in multipliers or not math.isfinite(number):\n        raise ValueError(f"Invalid unit for {flag_name} ({value!r})")\n    return int(round(number * multipliers[unit]))\n\n\ndef validate_ollama_version(value: str) -> str:\n    cleaned = str(value).strip().lstrip("v")\n    if not VERSION_RE.fullmatch(cleaned):\n        raise ValueError(f"Invalid Ollama version {value!r}: e.g. 0.17.7 or 0.18.0-rc1")\n    return cleaned\n\n\ndef parse_extra_ollama_env(entries: list[str] | None) -> dict[str, str]:\n    """Validate repeatable ``KEY=VALUE`` overrides for ``ollama serve``."""\n    out: dict[str, str] = {}\n    for raw in entries or ():\n        key, sep, val = str(raw).partition("=")\n        key = key.strip()\n        if not sep or not key:\n            raise ValueError(f"--ollama-env must have the form KEY=VALUE, got: {raw!r}")\n        if key in FORBIDDEN_ENV_KEYS or key.upper() in FORBIDDEN_ENV_KEYS:\n            raise ValueError(\n                f"{key} cannot be overridden with --ollama-env: OLLAMA_HOST must stay bound to 127.0.0.1, and proxies are configured with HTTPS_PROXY"\n            )\n        if not (key.startswith("OLLAMA_") or key in ALLOWED_EXTRA_ENV_EXACT or key.startswith("HSA_OVERRIDE_GFX_VERSION_")):\n            raise ValueError(\n                f"--ollama-env only accepts OLLAMA_* or supported GPU/proxy environment variables, got: {key!r}"\n            )\n        if "\\x00" in val or "\\n" in val or "\\r" in val:\n            raise ValueError(f"The value of --ollama-env {key} cannot contain newlines or null bytes")\n        out[key] = val\n    return out\n\n\ndef build_ollama_server_env(\n    *,\n    profile: str | None = None,\n    context_length: int | None = None,\n    flash_attention: bool | None = None,\n    kv_cache_type: str | None = None,\n    num_parallel: int | None = None,\n    max_loaded_models: int | None = None,\n    max_queue: int | None = None,\n    keep_alive: str | None = None,\n    load_timeout: str | None = None,\n    gpu_overhead: str | int | None = None,\n    sched_spread: bool | None = None,\n    max_transfer_streams: int | None = None,\n    origins: str | None = None,\n    no_cloud: bool | None = None,\n    ollama_debug: str | None = None,\n    extra_env: list[str] | None = None,\n) -> dict[str, str]:\n    """Build validated ``OLLAMA_*`` server environment variables (excluding host/models path)."""\n    env: dict[str, str] = {"OLLAMA_KEEP_ALIVE": DEFAULT_KEEP_ALIVE}\n    if profile is not None:\n        prof = profile.strip().lower()\n        if prof not in OPTIMIZATION_PROFILES:\n            raise ValueError(f"Unsupported optimization profile {profile!r}; choices: {\', \'.join(OPTIMIZATION_PROFILES)}")\n        env.update(OPTIMIZATION_PROFILES[prof])\n\n    if context_length is not None:\n        if int(context_length) < 256:\n            raise ValueError("--context-length must be >= 256")\n        env["OLLAMA_CONTEXT_LENGTH"] = str(int(context_length))\n\n    if kv_cache_type is not None:\n        kvt = kv_cache_type.strip().lower()\n        if kvt not in KV_CACHE_TYPES:\n            raise ValueError(f"--kv-cache-type must be one of {\', \'.join(KV_CACHE_TYPES)}")\n        if kvt in ("q8_0", "q4_0") and flash_attention is False:\n            raise ValueError(f"KV cache quantization ({kvt}) requires Flash Attention; it cannot be combined with --no-flash-attention")\n        env["OLLAMA_KV_CACHE_TYPE"] = kvt\n        if kvt in ("q8_0", "q4_0") and flash_attention is None:\n            # Ollama requires Flash Attention when quantizing the K/V cache.\n            env["OLLAMA_FLASH_ATTENTION"] = "1"\n\n    if flash_attention is not None:\n        env["OLLAMA_FLASH_ATTENTION"] = "1" if flash_attention else "0"\n        if not flash_attention and env.get("OLLAMA_KV_CACHE_TYPE") in ("q8_0", "q4_0"):\n            # Explicit --no-flash-attention overrides a preset\'s q8_0 KV cache back to f16 unless user also forced kv_cache_type\n            env["OLLAMA_KV_CACHE_TYPE"] = "f16"\n\n    if num_parallel is not None:\n        if int(num_parallel) < 1:\n            raise ValueError("--num-parallel must be >= 1")\n        env["OLLAMA_NUM_PARALLEL"] = str(int(num_parallel))\n\n    if max_loaded_models is not None:\n        if int(max_loaded_models) < 0:\n            raise ValueError("--max-loaded-models must be >= 0 (0 means automatic)")\n        env["OLLAMA_MAX_LOADED_MODELS"] = str(int(max_loaded_models))\n\n    if max_queue is not None:\n        if int(max_queue) < 1:\n            raise ValueError("--max-queue must be >= 1")\n        env["OLLAMA_MAX_QUEUE"] = str(int(max_queue))\n\n    if keep_alive is not None:\n        env["OLLAMA_KEEP_ALIVE"] = validate_duration(keep_alive, "--keep-alive")\n\n    if load_timeout is not None:\n        env["OLLAMA_LOAD_TIMEOUT"] = validate_duration(load_timeout, "--load-timeout")\n\n    if gpu_overhead is not None:\n        env["OLLAMA_GPU_OVERHEAD"] = str(parse_byte_size(gpu_overhead, "--gpu-overhead"))\n\n    if sched_spread is not None:\n        env["OLLAMA_SCHED_SPREAD"] = "1" if sched_spread else "0"\n\n    if max_transfer_streams is not None:\n        if int(max_transfer_streams) < 1:\n            raise ValueError("--max-transfer-streams must be >= 1")\n        env["OLLAMA_MAX_TRANSFER_STREAMS"] = str(int(max_transfer_streams))\n\n    if origins is not None:\n        cleaned_origins = origins.strip()\n        if "\\n" in cleaned_origins or "\\r" in cleaned_origins or "\\x00" in cleaned_origins:\n            raise ValueError("--origins cannot contain newlines or null bytes")\n        env["OLLAMA_ORIGINS"] = cleaned_origins\n\n    if no_cloud is not None:\n        env["OLLAMA_NO_CLOUD"] = "1" if no_cloud else "0"\n\n    if ollama_debug is not None:\n        dbg = str(ollama_debug).strip().lower()\n        if dbg in ("0", "false", "off"):\n            env["OLLAMA_DEBUG"] = "0"\n        elif dbg in ("1", "true", "on", "debug"):\n            env["OLLAMA_DEBUG"] = "1"\n        elif dbg in ("2", "trace"):\n            env["OLLAMA_DEBUG"] = "2"\n        else:\n            raise ValueError("--ollama-debug accepts 0, 1 (debug) or 2 (trace)")\n\n    env.update(parse_extra_ollama_env(extra_env))\n    return env\n\n\ndef allowed_caddy_paths(compat_apis: tuple[str, ...]) -> list[str]:\n    """Return deduplicated Caddy path matchers for the selected API families (always keeping `/` health)."""\n    paths: list[str] = ["/"]\n    for api in compat_apis:\n        for pat in API_FAMILY_PATHS.get(api, ()):\n            if pat not in paths:\n                paths.append(pat)\n    return paths\n\n\ndef _escape_caddy_quoted(value: str, label: str) -> str:\n    if any(ch in value for ch in ("\\r", "\\n", "\\x00", \'"\', "\\\\")):\n        raise ValueError(f"Unsafe character in Caddy {label}")\n    return value\n\n\ndef render_ollama_caddyfile(\n    *,\n    public_port: int,\n    service_port: int,\n    auth_user: str,\n    hashed_pass: str,\n    api_token: str,\n    cookie_tok: str,\n    compat_apis: tuple[str, ...] = COMPAT_APIS,\n    read_only_models: bool = False,\n) -> str:\n    """Render Caddyfile supporting Bearer, Anthropic X-Api-Key, browser Basic Auth, and API family filtering."""\n    safe_user = _escape_caddy_quoted(auth_user, "auth_user")\n    safe_token = _escape_caddy_quoted(api_token, "api_token")\n    safe_cookie = _escape_caddy_quoted(cookie_tok, "cookie_tok")\n    compat = parse_compat_apis(list(compat_apis))\n\n    matcher_defs: list[str] = [\n        f\'    @bearer header Authorization "Bearer {safe_token}"\',\n        f\'    @apikey header X-Api-Key "{safe_token}"\',\n        f"    @cookie header Cookie *ct_session={safe_cookie}*",\n    ]\n    guard_lines: list[str] = []\n\n    if read_only_models:\n        matcher_defs.append(f"    @blocked_mgmt path {\' \'.join(MODEL_MUTATION_PATHS)}")\n        guard_lines.extend([\n            \'            header @blocked_mgmt Content-Type "application/json"\',\n            \'            respond @blocked_mgmt "{\\\\"error\\\\":\\\\"model management endpoints are disabled (--read-only-models)\\\\"}" 403\',\n        ])\n\n    if set(compat) != set(COMPAT_APIS):\n        allowed = " ".join(allowed_caddy_paths(compat))\n        matcher_defs.append(f"    @disallowed_api not path {allowed}")\n        enabled_str = ",".join(compat)\n        guard_lines.extend([\n            \'            header @disallowed_api Content-Type "application/json"\',\n            f\'            respond @disallowed_api "{{\\\\"error\\\\":\\\\"API endpoint is not enabled (active compat: {enabled_str})\\\\"}}" 403\',\n        ])\n\n    proxy_block = (\n        f"            reverse_proxy 127.0.0.1:{int(service_port)} {{\\n"\n        f"                header_up Host {{upstream_hostport}}\\n"\n        f"                header_up Origin http://{{upstream_hostport}}\\n"\n        f"                header_up -Authorization\\n"\n        f"                header_up -X-Api-Key\\n"\n        f"                flush_interval -1\\n"\n        f"            }}"\n    )\n    route_inner = "\\n".join([*guard_lines, proxy_block])\n    authenticated_route = f"        route {{\\n{route_inner}\\n        }}"\n    matchers_text = "\\n".join(matcher_defs)\n\n    return f"""{{\n    admin off\n    auto_https off\n}}\n:{int(public_port)} {{\n    encode zstd gzip\n{matchers_text}\n    handle @bearer {{\n{authenticated_route}\n    }}\n    handle @apikey {{\n{authenticated_route}\n    }}\n    handle @cookie {{\n{authenticated_route}\n    }}\n    handle {{\n        route {{\n            basic_auth {{\n                "{safe_user}" {hashed_pass}\n            }}\n            header +Set-Cookie "ct_session={safe_cookie}; Path=/; HttpOnly; SameSite=Lax"\n{route_inner}\n        }}\n    }}\n}}\n"""\n\n\ndef compat_endpoints_info(\n    base_url: str,\n    compat_apis: tuple[str, ...] = COMPAT_APIS,\n    *,\n    api_token: str | None = None,\n    read_only_models: bool = False,\n) -> dict[str, Any]:\n    """Build structured metadata showing active compatible APIs, Base URLs, and SDK settings."""\n    clean_url = (base_url or "").rstrip("/")\n    compat = parse_compat_apis(list(compat_apis))\n    token_display = api_token if api_token else "<API_TOKEN>"\n    interfaces: dict[str, Any] = {}\n    if "ollama" in compat:\n        endpoints = ["/api/chat", "/api/generate", "/api/embed", "/api/tags", "/api/ps", "/api/show", "/api/version"]\n        if not read_only_models:\n            endpoints.extend(["/api/pull", "/api/copy", "/api/create", "/api/delete"])\n        interfaces["ollama"] = {\n            "base_url": clean_url,\n            "auth_header": f"Authorization: Bearer {token_display}",\n            "read_only_models": read_only_models,\n            "endpoints": endpoints,\n        }\n    if "openai" in compat:\n        interfaces["openai"] = {\n            "base_url": f"{clean_url}/v1" if clean_url else "/v1",\n            "api_key": token_display,\n            "auth_header": f"Authorization: Bearer {token_display}",\n            "endpoints": [\n                "/v1/chat/completions",\n                "/v1/responses",\n                "/v1/completions",\n                "/v1/embeddings",\n                "/v1/models",\n                "/v1/audio/speech",\n                "/v1/audio/transcriptions",\n                "/v1/audio/translations",\n            ],\n        }\n    if "anthropic" in compat:\n        interfaces["anthropic"] = {\n            "base_url": clean_url,\n            "api_key": token_display,\n            "auth_headers": [\n                f"Authorization: Bearer {token_display}",\n                f"x-api-key: {token_display}",\n            ],\n            "env": {\n                "ANTHROPIC_BASE_URL": clean_url,\n                "ANTHROPIC_API_KEY": token_display,\n            },\n            "endpoints": ["/v1/messages", "/v1/models"],\n        }\n    return {\n        "url": clean_url or None,\n        "compat": list(compat),\n        "read_only_models": read_only_models,\n        "interfaces": interfaces,\n    }\n\n\ndef pull_model_stream(\n    api_base: str,\n    model: str,\n    *,\n    headers: dict[str, str] | None = None,\n    opener: Callable[..., Any] | None = None,\n    log_fn: Callable[[str], None] | None = None,\n    timeout: float | None = 1800,\n    retries: int = 3,\n) -> dict[str, Any]:\n    """Stream ``POST /api/pull`` with throttled progress logging and automatic resume retries."""\n    spec = validate_model_spec(model)\n    open_url = opener or urllib.request.urlopen\n    req_headers = {"Content-Type": "application/json", "User-Agent": "Mozilla/5.0"}\n    if headers:\n        req_headers.update(headers)\n    payload = json.dumps({"model": spec, "stream": True}).encode("utf-8")\n    url = f"{api_base.rstrip(\'/\')}/api/pull"\n\n    last_err: Exception | None = None\n    for attempt in range(1, max(1, retries) + 1):\n        req = urllib.request.Request(url, data=payload, headers=req_headers, method="POST")\n        last_log_t = 0.0\n        last_status = ""\n        final_event: dict[str, Any] = {}\n        try:\n            with open_url(req, timeout=timeout) as resp:\n                for raw_line in resp:\n                    line = raw_line.decode("utf-8", errors="replace").strip()\n                    if not line:\n                        continue\n                    event = json.loads(line)\n                    if event.get("error"):\n                        raise RuntimeError(f"Pulling model {spec} failed: {event[\'error\']}")\n                    final_event = event\n                    status = str(event.get("status") or "")\n                    total = event.get("total")\n                    completed = event.get("completed")\n                    now = time.monotonic()\n                    if log_fn:\n                        if isinstance(total, int) and total > 0 and isinstance(completed, int):\n                            pct = completed * 100.0 / total\n                            if now - last_log_t >= 5.0 or (status != last_status and pct in (0.0, 100.0)):\n                                log_fn(\n                                    f"pulling {spec}: {status} {pct:.1f}% "\n                                    f"({completed / 1e9:.2f}/{total / 1e9:.2f} GB)"\n                                )\n                                last_log_t = now\n                                last_status = status\n                        elif status and status != last_status:\n                            log_fn(f"pulling {spec}: {status}")\n                            last_status = status\n                            last_log_t = now\n            return {"model": spec, "status": final_event.get("status", "success")}\n        except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as err:\n            # HTTP 4xx errors (e.g. 401/403/404 invalid model) should fail fast without retrying\n            if isinstance(err, urllib.error.HTTPError) and 400 <= err.code < 500 and err.code != 429:\n                body = err.read().decode("utf-8", errors="replace")\n                raise RuntimeError(f"HTTP {err.code} pulling {spec}: {body}") from None\n            last_err = err\n            if attempt < retries:\n                if log_fn:\n                    log_fn(f"pull {spec} interrupted ({err}); retrying ({attempt + 1}/{retries})...")\n                time.sleep(min(2 * attempt, 6))\n                continue\n            raise RuntimeError(f"Pulling model {spec} failed (after {retries} retries): {err}") from None\n    raise RuntimeError(f"Pulling model {spec} failed: {last_err}")\n\n\ndef copy_model_alias(\n    api_base: str,\n    source: str,\n    destination: str,\n    *,\n    headers: dict[str, str] | None = None,\n    opener: Callable[..., Any] | None = None,\n    timeout: float | None = 60,\n) -> dict[str, str]:\n    """Create a model alias via ``POST /api/copy``."""\n    src = validate_model_spec(source)\n    dst = validate_model_spec(destination)\n    open_url = opener or urllib.request.urlopen\n    req_headers = {"Content-Type": "application/json", "User-Agent": "Mozilla/5.0"}\n    if headers:\n        req_headers.update(headers)\n    payload = json.dumps({"source": src, "destination": dst}).encode("utf-8")\n    req = urllib.request.Request(f"{api_base.rstrip(\'/\')}/api/copy", data=payload, headers=req_headers, method="POST")\n    try:\n        with open_url(req, timeout=timeout) as resp:\n            resp.read()\n    except urllib.error.HTTPError as err:\n        body = err.read().decode("utf-8", errors="replace")\n        raise RuntimeError(f"HTTP {err.code} copying model {src} -> {dst}: {body}") from None\n    return {"source": src, "alias": dst, "status": "copied"}\n\n\ndef preload_or_unload_model(\n    api_base: str,\n    model: str,\n    *,\n    keep_alive: str | int | None = None,\n    headers: dict[str, str] | None = None,\n    opener: Callable[..., Any] | None = None,\n    timeout: float | None = 600,\n) -> dict[str, Any]:\n    """Preload a model into VRAM (or unload with ``keep_alive=0``), falling back to ``/api/embed`` for embedding models."""\n    spec = validate_model_spec(model)\n    open_url = opener or urllib.request.urlopen\n    req_headers = {"Content-Type": "application/json", "User-Agent": "Mozilla/5.0"}\n    if headers:\n        req_headers.update(headers)\n\n    gen_body: dict[str, Any] = {"model": spec}\n    normalized_ka = normalize_api_keep_alive(keep_alive)\n    if normalized_ka is not None:\n        gen_body["keep_alive"] = normalized_ka\n\n    gen_req = urllib.request.Request(\n        f"{api_base.rstrip(\'/\')}/api/generate",\n        data=json.dumps(gen_body).encode("utf-8"),\n        headers=req_headers,\n        method="POST",\n    )\n    try:\n        with open_url(gen_req, timeout=timeout) as resp:\n            raw = resp.read().decode("utf-8", errors="replace").strip()\n            data = json.loads(raw) if raw else {}\n            return {"model": spec, "endpoint": "/api/generate", "done_reason": data.get("done_reason", "load")}\n    except urllib.error.HTTPError as err:\n        body = err.read().decode("utf-8", errors="replace")\n        # Embedding-only models return HTTP 400 "does not support generate" -> preload via /api/embed\n        if err.code == 400 and ("embed" in body.lower() or "does not support generate" in body.lower()):\n            emb_body: dict[str, Any] = {"model": spec, "input": "" if normalized_ka == 0 else "warmup"}\n            if normalized_ka is not None:\n                emb_body["keep_alive"] = normalized_ka\n            emb_req = urllib.request.Request(\n                f"{api_base.rstrip(\'/\')}/api/embed",\n                data=json.dumps(emb_body).encode("utf-8"),\n                headers=req_headers,\n                method="POST",\n            )\n            try:\n                with open_url(emb_req, timeout=timeout) as resp2:\n                    resp2.read()\n                    return {"model": spec, "endpoint": "/api/embed", "done_reason": "load"}\n            except urllib.error.HTTPError as err2:\n                body2 = err2.read().decode("utf-8", errors="replace")\n                raise RuntimeError(f"HTTP {err2.code} preloading embedding model {spec}: {body2}") from None\n        raise RuntimeError(f"HTTP {err.code} preloading model {spec}: {body}") from None\n', encoding="utf-8")
if WORK not in sys.path:
    sys.path.insert(0, WORK)

from ollama_runtime import (
    COMPAT_APIS,
    build_ollama_server_env,
    compat_endpoints_info,
    copy_model_alias,
    models_to_pull,
    parse_compat_apis,
    parse_model_aliases,
    parse_model_specs,
    preload_or_unload_model,
    pull_model_stream,
    render_ollama_caddyfile,
    validate_ollama_version,
)

def olog(msg):
    print(time.strftime("[%F %T] [ollama] ") + msg, flush=True)

OLLAMA_NO_MODELS = os.environ.get("CT_OLLAMA_NO_MODELS", "0") == "1"
OLLAMA_MODELS_LIST = parse_model_specs(os.environ.get("CT_MODELS"), no_models=OLLAMA_NO_MODELS)
OLLAMA_ALIASES = parse_model_aliases(os.environ.get("CT_OLLAMA_ALIASES"))
OLLAMA_PULL_LIST = models_to_pull(OLLAMA_MODELS_LIST, OLLAMA_ALIASES)
OLLAMA_PRELOAD_MODE = os.environ.get("CT_OLLAMA_PRELOAD", "primary").strip().lower()
OLLAMA_COMPAT = parse_compat_apis(os.environ.get("CT_OLLAMA_COMPAT"))
OLLAMA_READ_ONLY_MODELS = os.environ.get("CT_OLLAMA_READ_ONLY_MODELS", "0") == "1"
OLLAMA_STORAGE_MODE = os.environ.get("CT_OLLAMA_MODELS_STORAGE", "ssd").strip().lower()
OLLAMA_MODELS_DIR = os.environ.get("CT_OLLAMA_MODELS") or (
    f"{DATA_DIR}/models" if OLLAMA_STORAGE_MODE == "drive" else "/content/ollama/models"
)
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)

OLLAMA_REQUESTED_VERSION = (
    validate_ollama_version(os.environ["CT_OLLAMA_VERSION"])
    if os.environ.get("CT_OLLAMA_VERSION")
    else ""
)
_installed_ver = (
    subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
    if shutil.which("ollama")
    else ""
)
if not shutil.which("ollama") or (OLLAMA_REQUESTED_VERSION and OLLAMA_REQUESTED_VERSION not in _installed_ver):
    olog("Installing Ollama" + (f" v{OLLAMA_REQUESTED_VERSION}" if OLLAMA_REQUESTED_VERSION else "") + " ...")
    _inst_env = dict(os.environ)
    if OLLAMA_REQUESTED_VERSION:
        _inst_env["OLLAMA_VERSION"] = OLLAMA_REQUESTED_VERSION
    subprocess.run(
        "apt-get install -y -qq zstd pciutils lshw >/dev/null 2>&1; "
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True,
        check=True,
        env=_inst_env,
    )
# Ensure systemd does not hold :11434 with default settings instead of our tuned OLLAMA_ENV
subprocess.run("systemctl stop ollama >/dev/null 2>&1 || true; systemctl disable ollama >/dev/null 2>&1 || true",
               shell=True, check=False)
olog(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())


In [ ]:
# @title 2. Service -- ollama serve on 127.0.0.1 + automatic model pull, alias & GPU VRAM preload
if "OLLAMA_COMPAT" not in globals():
    raise RuntimeError("Ollama setup cell failed before completing initialization; see traceback above.")

_server_env_raw = os.environ.get("CT_OLLAMA_SERVER_ENV", "").strip()
OLLAMA_ENV = json.loads(_server_env_raw) if _server_env_raw else build_ollama_server_env()
for _k, _v in os.environ.items():
    if _k.startswith("OLLAMA_") and _k not in ("OLLAMA_HOST", "OLLAMA_MODELS") and _k not in OLLAMA_ENV:
        OLLAMA_ENV[_k] = _v
OLLAMA_ENV["OLLAMA_HOST"] = f"127.0.0.1:{SERVICE_PORT}"
OLLAMA_ENV["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR

olog(f"GPU: {ct_gpu_stat('name,memory.total,compute_cap')} | storage={OLLAMA_STORAGE_MODE} ({OLLAMA_MODELS_DIR})")
olog(f"Compatible APIs: {','.join(OLLAMA_COMPAT)} | read_only_models={OLLAMA_READ_ONLY_MODELS}")
olog("Server env: " + ", ".join(f"{k}={v}" for k, v in sorted(OLLAMA_ENV.items())))

def start_service():
    subprocess.run(["pkill", "-f", "ollama serve"], check=False)
    return ct_spawn("service", ["ollama", "serve"], env=OLLAMA_ENV)

def _wait_ollama_local(timeout=120):
    deadline = time.time() + timeout
    url = f"http://127.0.0.1:{SERVICE_PORT}/api/version"
    while time.time() < deadline:
        proc = PROCS["service"][0]
        if proc.poll() is not None:
            raise RuntimeError(f"ollama serve exited early (code {proc.returncode}):\n{ct_tail('service', 50)}")
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                if resp.status == 200:
                    return
        except Exception:
            pass
        time.sleep(1.5)
    raise TimeoutError(f"ollama serve not ready after {timeout}s:\n{ct_tail('service', 50)}")

PROCS["service"] = (start_service(), start_service)
_wait_ollama_local(timeout=120)

LOCAL_OLLAMA_URL = f"http://127.0.0.1:{SERVICE_PORT}"
for _m in OLLAMA_PULL_LIST:
    pull_model_stream(LOCAL_OLLAMA_URL, _m, log_fn=olog, timeout=1800, retries=3)

for _alias_name, _src_model in OLLAMA_ALIASES:
    copy_model_alias(LOCAL_OLLAMA_URL, _src_model, _alias_name, timeout=60)
    olog(f"Created model alias: {_alias_name} -> {_src_model}")

_preload_targets = []
if OLLAMA_PRELOAD_MODE == "all":
    _preload_targets = list(OLLAMA_MODELS_LIST or [a for a, _ in OLLAMA_ALIASES])
elif OLLAMA_PRELOAD_MODE != "none":
    _candidates = list(OLLAMA_MODELS_LIST or [a for a, _ in OLLAMA_ALIASES])
    if _candidates:
        _preload_targets = [_candidates[0]]

for _pm in _preload_targets:
    olog(f"Preloading {_pm} into GPU memory (keep_alive={OLLAMA_ENV.get('OLLAMA_KEEP_ALIVE')}) ...")
    _preload_res = preload_or_unload_model(
        LOCAL_OLLAMA_URL,
        _pm,
        keep_alive=OLLAMA_ENV.get("OLLAMA_KEEP_ALIVE"),
        timeout=600,
    )
    olog(f"Preloaded {_pm} via {_preload_res['endpoint']}")

print(subprocess.run(["ollama", "list"], env={**os.environ, **OLLAMA_ENV}, capture_output=True, text=True).stdout)
print(subprocess.run(["ollama", "ps"], env={**os.environ, **OLLAMA_ENV}, capture_output=True, text=True).stdout)


In [ ]:
# @title 3. Auth & compatible API routing -- Caddy (Basic Auth + Bearer + Anthropic X-Api-Key + --compat filter)
import hashlib, platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    caddyfile = render_ollama_caddyfile(
        public_port=PUBLIC_PORT,
        service_port=SERVICE_PORT,
        auth_user=AUTH_USER,
        hashed_pass=hashed,
        api_token=API_TOKEN,
        cookie_tok=cookie_tok,
        compat_apis=OLLAMA_COMPAT,
        read_only_models=OLLAMA_READ_ONLY_MODELS,
    )
    open(f"{WORK}/Caddyfile", "w", encoding="utf-8").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    olog(f"Caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (Bearer + X-Api-Key + Basic Auth, compat={','.join(OLLAMA_COMPAT)})")
else:
    AUTH_OK = True
    olog(f"auth={AUTH}: targeting :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- verify Bearer, X-Api-Key, Basic Auth & compatible API policy before CT_READY
import base64

if "OLLAMA_COMPAT" not in globals() or "service" not in PROCS:
    raise RuntimeError("Ollama service did not start because an earlier notebook cell failed; see traceback above.")

ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/api/version", timeout=120, ok=lambda c: c == 200)

def _expect_http(url, *, headers=None, method="GET", body=None, expected=401, label="probe"):
    req = urllib.request.Request(url, data=body, headers={"User-Agent": "Mozilla/5.0", **(headers or {})}, method=method)
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            if resp.status != expected:
                raise RuntimeError(f"{label}: expected HTTP {expected}, got HTTP {resp.status}")
            return resp
    except urllib.error.HTTPError as err:
        if err.code != expected:
            raise RuntimeError(f"{label}: expected HTTP {expected}, got HTTP {err.code}")
        return err

def _public_bearer_probe():
    req = urllib.request.Request(
        PUBLIC_URL + "/",
        headers={"Authorization": f"Bearer {API_TOKEN}", "User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(req, timeout=20) as resp:
        return resp.status

def _verify_public_auth():
    if AUTH != "caddy":
        return
    # 1. Unauthenticated request must return 401 without Set-Cookie
    err_unauth = _expect_http(PUBLIC_URL + "/", expected=401, label="unauthenticated /")
    if getattr(err_unauth, "headers", None) and err_unauth.headers.get("Set-Cookie"):
        raise RuntimeError("Public 401 response leaked Set-Cookie")
    # 2. Invalid Bearer & invalid X-Api-Key must return 401
    _expect_http(PUBLIC_URL + "/", headers={"Authorization": "Bearer invalid-probe-token"}, expected=401, label="invalid Bearer")
    _expect_http(PUBLIC_URL + "/", headers={"X-Api-Key": "invalid-probe-token"}, expected=401, label="invalid X-Api-Key")
    # 3. Valid Anthropic-style X-Api-Key must succeed
    _expect_http(PUBLIC_URL + "/", headers={"X-Api-Key": API_TOKEN}, expected=200, label="valid X-Api-Key")
    # 4. Valid Basic Auth must succeed
    basic = base64.b64encode(f"{AUTH_USER}:{AUTH_PASS}".encode("utf-8")).decode("ascii")
    _expect_http(PUBLIC_URL + "/", headers={"Authorization": f"Basic {basic}"}, expected=200, label="valid Basic Auth")
    # 5. Disabled API families & read-only model guard must return 403
    auth_hdr = {"Authorization": f"Bearer {API_TOKEN}", "Content-Type": "application/json"}
    if "ollama" not in OLLAMA_COMPAT:
        _expect_http(PUBLIC_URL + "/api/tags", headers=auth_hdr, expected=403, label="disabled ollama /api/tags")
    elif OLLAMA_READ_ONLY_MODELS:
        _expect_http(PUBLIC_URL + "/api/pull", headers=auth_hdr, method="POST", body=b"{}", expected=403, label="read-only /api/pull")
    if "openai" not in OLLAMA_COMPAT:
        _expect_http(PUBLIC_URL + "/v1/chat/completions", headers=auth_hdr, method="POST", body=b"{}", expected=403, label="disabled openai /v1/chat/completions")
    if "anthropic" not in OLLAMA_COMPAT:
        _expect_http(PUBLIC_URL + "/v1/messages", headers={"X-Api-Key": API_TOKEN, "Content-Type": "application/json"},
                     method="POST", body=b"{}", expected=403, label="disabled anthropic /v1/messages")

ct_public_gate(PUBLIC_URL, _public_bearer_probe, _verify_public_auth, olog, timeout=180)
if AUTH == "caddy" and not CREDS_FROM_ENV:
    print(f"CT_CREDENTIALS user={AUTH_USER} pass={AUTH_PASS}")
    print(f"API token: {API_TOKEN}")
for _api_name, _detail in compat_endpoints_info(PUBLIC_URL, OLLAMA_COMPAT, read_only_models=OLLAMA_READ_ONLY_MODELS)["interfaces"].items():
    olog(f"[{_api_name}] base_url={_detail['base_url']} routes={','.join(_detail['endpoints'])}")
print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

def _ollama_health():
    if PROCS["service"][0].poll() is not None:
        return False
    with urllib.request.urlopen(f"http://127.0.0.1:{SERVICE_PORT}/api/version", timeout=10) as resp:
        return resp.status == 200

def _ollama_heartbeat(uptime_s):
    if uptime_s % 600 < 60:
        states = " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items())
        olog(f"[heartbeat] uptime={uptime_s}s {states} gpu={ct_gpu_stat()}")

ct_watchdog(_ollama_health, start_service, olog, interval=60, grace=300, heartbeat=_ollama_heartbeat)
